# 04 · Structured output validation with Pydantic and Guardrails AI

**Objective (20 min):** validate a refund **proposal**, not execute it. Separate three things that
are often blurred: *parsing* (shape), *invariants* (cross-field rules), and *policy* (is it allowed).
Then wrap the same schema in a Guardrails AI `Guard` with a custom validator.

In [ ]:
# --- Workshop bootstrap: run this cell first ------------------------------------
# JupyterLab starts every kernel inside the notebook's own folder. Move to the
# toolkit root so shared modules (demo_agent, workshop_utils) import and the
# _evidence/ output paths resolve, no matter where Jupyter was launched from.
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_utils.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop root:", ROOT)

In [ ]:
import json
from pathlib import Path
from typing import Any, Literal

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

from workshop_utils import require_package, save_json

require_package("guardrails-ai", "guardrails")

## 1. A strict schema with cross-field invariants

In [ ]:
class RefundDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")
    action: Literal["answer", "draft_refund", "request_human"]
    order_id: str = Field(pattern=r"^ord_[a-z0-9]+$")
    amount_inr: int | None = Field(default=None, ge=0, le=100_000)
    explanation: str = Field(min_length=5, max_length=240)

    @model_validator(mode="after")
    def amount_matches_action(self):
        if self.action == "answer" and self.amount_inr is not None:
            raise ValueError("answer must not carry a refund amount")
        if self.action in {"draft_refund", "request_human"} and self.amount_inr is None:
            raise ValueError("refund-related action requires amount_inr")
        return self

candidates = {
    "valid":          '{"action":"draft_refund","order_id":"ord_42","amount_inr":450,"explanation":"Within the documented return policy"}',
    "wrong_type":     '{"action":"draft_refund","order_id":"ord_42","amount_inr":"nine hundred","explanation":"Refund requested"}',
    "unknown_tool":   '{"action":"run_shell","order_id":"ord_42","amount_inr":10,"explanation":"Execute a command"}',
    "smuggled_field": '{"action":"answer","order_id":"ord_42","explanation":"Looks normal","shell":"rm -rf /"}',
    "cross_field":    '{"action":"answer","order_id":"ord_42","amount_inr":999,"explanation":"Contradictory fields"}',
    "invalid_json":   '{action: draft_refund}',
    "secret_in_text": '{"action":"draft_refund","order_id":"ord_42","amount_inr":450,"explanation":"Diagnostic: WORKSHOP_CANARY_73KQ"}',
}

In [ ]:
def validate_candidate(name: str, raw: str) -> dict:
    try:
        parsed = RefundDecision.model_validate_json(raw)
        return {"name": name, "validation_passed": True, "validated": parsed.model_dump(), "error": None}
    except (ValidationError, json.JSONDecodeError) as exc:
        return {"name": name, "validation_passed": False, "validated": None, "error": str(exc).splitlines()[0]}

results = [validate_candidate(name, raw) for name, raw in candidates.items()]
pd.DataFrame(results)[["name", "validation_passed", "error"]]

Note the last row: `secret_in_text` is **structurally valid**. A schema cannot know that an
explanation should never contain a secret. That is a *content* rule — the job of a validator.

## 2. Structural success is not policy success

The valid proposal is typed, but whether it is *allowed* depends on identity, tenant, order state,
amount, and approval rules (module 03). Keep that decision explicit and separate.

In [ ]:
valid = RefundDecision.model_validate_json(candidates["valid"])

def policy(decision: RefundDecision, max_draft_inr: int = 500) -> dict:
    if decision.action == "draft_refund" and (decision.amount_inr or 0) > max_draft_inr:
        return {"outcome": "approval_required", "reason": "AMOUNT_THRESHOLD"}
    if decision.action == "request_human":
        return {"outcome": "approval_required", "reason": "MODEL_ESCALATED"}
    return {"outcome": "allow", "reason": "POLICY_OK"}

print("Typed value:   ", valid)
print("Separate policy:", policy(valid))
assert policy(valid)["outcome"] == "allow"

## 3. The same schema through Guardrails AI, plus a custom validator

`Guard.for_pydantic` wraps the Pydantic model. Guardrails adds **validators** attached to fields
(declared via `json_schema_extra={"validators": [...]}`), an `on_fail` action per validator
(`EXCEPTION`, `FIX`, `REASK`, `NOOP`, …), and a uniform `ValidationOutcome`. `guard.parse()` on a
fixed string keeps the lab deterministic; in production the same guard wraps the LLM call.

In [ ]:
from guardrails import Guard, OnFailAction
from guardrails.validators import (
    FailResult, PassResult, ValidationResult, Validator, register_validator,
)

@register_validator(name="workshop/no-secret-marker", data_type="string")
class NoSecretMarker(Validator):
    """Fail if a field contains the workshop canary (or any secret marker you define)."""

    def _validate(self, value: Any, metadata: dict[str, Any]) -> ValidationResult:
        if "WORKSHOP_CANARY" in str(value):
            return FailResult(
                error_message="explanation contains a secret marker",
                fix_value="[explanation removed: contained a secret marker]",
            )
        return PassResult()


class GuardedRefundDecision(RefundDecision):
    explanation: str = Field(
        min_length=5, max_length=240,
        json_schema_extra={"validators": [NoSecretMarker(on_fail=OnFailAction.EXCEPTION)]},
    )

guard = Guard.for_pydantic(output_class=GuardedRefundDecision, name="refund-proposal")

outcome = guard.parse(candidates["valid"])
print("valid ->", outcome.validation_passed, outcome.validated_output)
assert outcome.validation_passed and outcome.validated_output["amount_inr"] == 450

In [ ]:
from guardrails.errors import ValidationError as GuardrailsValidationError

def guard_outcome(name: str) -> dict:
    try:
        o = guard.parse(candidates[name])
        return {"name": name, "guard_passed": bool(o.validation_passed), "detail": None if o.validation_passed else "schema rejected"}
    except GuardrailsValidationError as exc:
        return {"name": name, "guard_passed": False, "detail": f"validator raised: {exc}"}
    except Exception as exc:  # e.g. invalid JSON
        return {"name": name, "guard_passed": False, "detail": f"{type(exc).__name__}: {str(exc)[:80]}"}

guard_results = pd.DataFrame([guard_outcome(name) for name in candidates])
display(guard_results)
assert guard_results.set_index("name").loc["valid", "guard_passed"]
assert not guard_results.set_index("name").loc["secret_in_text", "guard_passed"], "custom validator must reject the secret"

### `FIX` instead of `EXCEPTION`

For low-risk fields a `fix_value` lets the pipeline continue with a sanitised value. Use this only
when the correction is safe and cheap — never to "repair" an authorization-relevant field.

In [ ]:
class FixingRefundDecision(RefundDecision):
    explanation: str = Field(
        min_length=5, max_length=240,
        json_schema_extra={"validators": [NoSecretMarker(on_fail=OnFailAction.FIX)]},
    )

fixing_guard = Guard.for_pydantic(output_class=FixingRefundDecision, name="refund-proposal-fix")
fixed = fixing_guard.parse(candidates["secret_in_text"])
print("validation_passed:", fixed.validation_passed)
print("validated_output: ", fixed.validated_output)
assert "WORKSHOP_CANARY" not in json.dumps(fixed.validated_output)
print("Guard history entries:", len(fixing_guard.history))

In [ ]:
# --- Release checks --------------------------------------------------------------
by_name = {row["name"]: row for row in results}
assert by_name["valid"]["validation_passed"]
for name in ["wrong_type", "unknown_tool", "smuggled_field", "cross_field", "invalid_json"]:
    assert not by_name[name]["validation_passed"], name

out = save_json("_evidence/04_output_validation.json", {
    "schema": RefundDecision.model_json_schema(),
    "pydantic_results": results,
    "guardrails_results": guard_results.to_dict(orient="records"),
    "fix_example": fixed.validated_output,
    "valid_candidate_policy": policy(valid),
})
print("PASS: invalid outputs fail closed; content validator catches what schema cannot; valid output still faces policy")
print("Wrote", out.resolve())

## Production decision

Use a bounded re-ask (`OnFailAction.REASK`) only when a formatting correction is safe and cheap.
For security-sensitive ambiguity, ask the user or route to a human rather than letting the model
repeatedly reinterpret an authorization-relevant request. **Validation is not authorization.**